# Prática Guiada — SQL na prática: Banco de dados de heróis

**Objetivo:** criar e gerenciar um banco de dados relacional utilizando comandos SQL.

Nesta prática, vamos utilizar o **SQLite**, um sistema de banco de dados relacional que funciona sem a necessidade de instalar ou configurar um servidor.

O foco é compreender e executar comandos SQL. O Python será utilizado apenas para conectar ao banco e apresentar os resultados.

## Etapa 1 — Conhecendo o Google Colab

O **Google Colab** é uma plataforma que permite escrever e executar código diretamente no navegador. O processamento ocorre em um ambiente de execução (runtime), que pode ser disponibilizado na nuvem.

### Como funciona?
- Um notebook é organizado em **células**.
- Células de texto apresentam explicações e instruções.
- Células de código executam comandos.
- Podemos executar as células individualmente ou em sequência.

### Atenção à permanência dos dados
O ambiente de execução do Colab é temporário. Quando ele é encerrado ou desconectado, arquivos armazenados apenas nesse ambiente podem ser perdidos.

Nesta atividade, o banco será criado como um arquivo (`marvel.db`) no ambiente de execução. Ao final, poderemos baixá-lo para guardar uma cópia. O notebook, por sua vez, pode ser salvo no Google Drive.

**Importante:** executar o código novamente pode recriar ou alterar o banco. Observe os comentários de cada etapa.

In [1]:
# Importa a biblioteca sqlite3, que permite trabalhar com bancos SQLite.
# Ela já faz parte da instalação padrão do Python no Colab.

import sqlite3

## Etapa 2 — Criando o banco de dados

No SQLite, o banco de dados é armazenado em um arquivo. A conexão abaixo cria o arquivo `marvel.db` caso ele ainda não exista.

A variável `conexao` representa a conexão com o banco. Por meio dela, enviaremos comandos SQL.

In [2]:
# Cria (ou abre, se já existir) o arquivo do banco de dados.
conexao = sqlite3.connect("marvel.db")

# Cria um cursor: objeto utilizado para executar comandos SQL.
cursor = conexao.cursor()

## Etapa 3 — Criando uma tabela

Vamos criar a tabela `Heroi`.

### Estrutura da tabela
- `id`: identificador único de cada herói.
- `nome`: nome do herói.
- `identidade`: identidade civil.
- `universo`: universo ou equipe de origem.
- `nivel_poder`: classificação numérica do nível de poder.

### Comando SQL: `CREATE TABLE`
`CREATE TABLE` cria uma nova tabela no banco de dados.

- `INTEGER`: número inteiro.
- `TEXT`: texto.
- `PRIMARY KEY`: chave primária, que identifica cada registro.
- `AUTOINCREMENT`: gera automaticamente novos valores para o identificador.

In [3]:
# Execute o comando SQL para criar a tabela.
# IF NOT EXISTS evita erro caso a tabela já exista.

cursor.execute("""
CREATE TABLE IF NOT EXISTS Heroi (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    identidade TEXT,
    universo TEXT,
    nivel_poder INTEGER
);
""")

# Confirma as alterações no banco.
conexao.commit()

## Etapa 4 — Inserindo registros

O comando `INSERT INTO` adiciona novos registros à tabela.

Observe que:
- Informamos os nomes das colunas.
- Os valores de texto ficam entre aspas.
- Não precisamos informar o `id`, pois ele é gerado automaticamente.
- O `?` é um parâmetro que será substituído pelo valor informado na tupla. Isso evita montar comandos SQL concatenando textos.

In [4]:
# Insere cinco heróis na tabela.

herois = [
    ("Homem de Ferro", "Tony Stark", "Marvel", 95),
    ("Homem-Aranha", "Peter Parker", "Marvel", 85),
    ("Capitão América", "Steve Rogers", "Marvel", 80),
    ("Thor", "Thor Odinson", "Marvel", 100),
    ("Hulk", "Bruce Banner", "Marvel", 98)
]

cursor.executemany(
    """
    INSERT INTO Heroi (nome, identidade, universo, nivel_poder)
    VALUES (?, ?, ?, ?);
    """,
    herois
)

# Confirma a inserção dos dados.
conexao.commit()

## Etapa 5 — Consultando os registros

O comando `SELECT` permite consultar dados armazenados.

### Consultar todas as colunas
`SELECT * FROM Heroi;`

O símbolo `*` significa que queremos todas as colunas da tabela.

In [5]:
# Consulta todos os registros da tabela Heroi.

cursor.execute("SELECT * FROM Heroi;")

# fetchall() retorna todos os resultados da consulta.
resultado = cursor.fetchall()

# Exibe os registros retornados.
for linha in resultado:
    print(linha)

(1, 'Homem de Ferro', 'Tony Stark', 'Marvel', 95)
(2, 'Homem-Aranha', 'Peter Parker', 'Marvel', 85)
(3, 'Capitão América', 'Steve Rogers', 'Marvel', 80)
(4, 'Thor', 'Thor Odinson', 'Marvel', 100)
(5, 'Hulk', 'Bruce Banner', 'Marvel', 98)


### Selecionando colunas específicas

Podemos escolher quais colunas serão exibidas, em vez de utilizar `*`.

In [6]:
# Altere a consulta para exibir somente nome e identidade.
# Complete o comando SQL no espaço indicado.

cursor.execute("SELECT nome, identidade FROM Heroi;")

resultado = cursor.fetchall()

for linha in resultado:
    print(linha)

('Homem de Ferro', 'Tony Stark')
('Homem-Aranha', 'Peter Parker')
('Capitão América', 'Steve Rogers')
('Thor', 'Thor Odinson')
('Hulk', 'Bruce Banner')


## Etapa 6 — Filtrando e ordenando resultados

### `WHERE`
O comando `WHERE` permite selecionar registros que atendam a uma condição.

### `ORDER BY`
O comando `ORDER BY` ordena os resultados.
- `ASC`: ordem crescente (padrão).
- `DESC`: ordem decrescente.

In [7]:
# DESAFIO 1
# Consulte os heróis cujo nivel_poder seja maior ou igual a 90.
# Complete a consulta SQL.

cursor.execute("""
SELECT *
FROM Heroi
WHERE nivel_poder > 90;
""")

for linha in cursor.fetchall():
    print(linha)

(1, 'Homem de Ferro', 'Tony Stark', 'Marvel', 95)
(4, 'Thor', 'Thor Odinson', 'Marvel', 100)
(5, 'Hulk', 'Bruce Banner', 'Marvel', 98)


In [8]:
# DESAFIO 2
# Exiba nome e nivel_poder de todos os heróis,
# ordenados do maior para o menor nivel_poder.

cursor.execute("""
SELECT nivel_poder
FROM Heroi
ORDER BY nivel_poder DESC;
""")

for linha in cursor.fetchall():
    print(linha)

(100,)
(98,)
(95,)
(85,)
(80,)


## Etapa 7 — Atualizando registros

O comando `UPDATE` modifica dados que já estão armazenados.

### Estrutura
`UPDATE tabela SET coluna = valor WHERE condição;`

**Atenção:** sem `WHERE`, todos os registros da tabela podem ser alterados.

In [9]:
# Exemplo: atualizar o nivel_poder do Homem-Aranha para 90.

cursor.execute("""
UPDATE Heroi
SET nivel_poder = 90
WHERE nome = 'Homem-Aranha';
""")

conexao.commit()

# Confira se a atualização ocorreu.
cursor.execute("SELECT nome, nivel_poder FROM Heroi;")

for linha in cursor.fetchall():
    print(linha)

('Homem de Ferro', 95)
('Homem-Aranha', 90)
('Capitão América', 80)
('Thor', 100)
('Hulk', 98)


## Etapa 8 — Excluindo registros

O comando `DELETE FROM` remove registros de uma tabela.

### Estrutura
`DELETE FROM tabela WHERE condição;`

**Atenção:** sem `WHERE`, todos os registros da tabela serão excluídos. A tabela continuará existindo.

In [10]:
# Exemplo: remover o Capitão América da tabela.

cursor.execute("""
DELETE FROM Heroi
WHERE nome = 'Capitão América';
""")

conexao.commit()

# Confira os registros restantes.
cursor.execute("SELECT * FROM Heroi;")

for linha in cursor.fetchall():
    print(linha)

(1, 'Homem de Ferro', 'Tony Stark', 'Marvel', 95)
(2, 'Homem-Aranha', 'Peter Parker', 'Marvel', 90)
(4, 'Thor', 'Thor Odinson', 'Marvel', 100)
(5, 'Hulk', 'Bruce Banner', 'Marvel', 98)


## Etapa 9 — Criando uma segunda tabela

Um banco relacional pode armazenar dados em diferentes tabelas e relacioná-las por meio de chaves.

Vamos criar a tabela `Equipe`. Cada equipe possui um identificador e um nome.

In [11]:
# Cria a tabela Equipe.

cursor.execute("""
CREATE TABLE IF NOT EXISTS Equipe (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL
);
""")

# Insere algumas equipes.
cursor.executemany(
    "INSERT INTO Equipe (nome) VALUES (?);",
    [
        ("Vingadores",),
        ("Defensores",),
        ("Guardiões da Galáxia",)
    ]
)

conexao.commit()

# Consulta as equipes cadastradas.
cursor.execute("SELECT * FROM Equipe;")

for linha in cursor.fetchall():
    print(linha)

(1, 'Vingadores')
(2, 'Defensores')
(3, 'Guardiões da Galáxia')


## Etapa 10 — Relacionando tabelas

Um herói pode participar de uma equipe. Para representar essa relação, vamos adicionar a coluna `equipe_id` à tabela `Heroi`.

Essa coluna será uma **chave estrangeira (FOREIGN KEY)**: seu valor referencia o `id` de um registro da tabela `Equipe`.

No SQLite, a verificação de chaves estrangeiras precisa ser habilitada na conexão.

In [12]:
# Habilita a verificação de chaves estrangeiras nesta conexão.
cursor.execute("PRAGMA foreign_keys = ON;")

# Cria uma nova tabela de relacionamento.
# Cada registro associa um herói a uma equipe.
cursor.execute("""
CREATE TABLE IF NOT EXISTS Heroi_Equipe (
    heroi_id INTEGER,
    equipe_id INTEGER,
    PRIMARY KEY (heroi_id, equipe_id),
    FOREIGN KEY (heroi_id) REFERENCES Heroi(id),
    FOREIGN KEY (equipe_id) REFERENCES Equipe(id)
);
""")

conexao.commit()

### Inserindo relações

Vamos associar heróis às equipes. Para isso, utilizamos os identificadores das tabelas, e não os nomes.

In [13]:
# Primeiro, consultamos os IDs para conferir os valores disponíveis.

cursor.execute("SELECT id, nome FROM Heroi;")
print("Heróis:")
for linha in cursor.fetchall():
    print(linha)

cursor.execute("SELECT id, nome FROM Equipe;")
print("Equipes:")
for linha in cursor.fetchall():
    print(linha)

Heróis:
(1, 'Homem de Ferro')
(2, 'Homem-Aranha')
(4, 'Thor')
(5, 'Hulk')
Equipes:
(1, 'Vingadores')
(2, 'Defensores')
(3, 'Guardiões da Galáxia')


In [14]:
# Vamos associar cada herói a uma equipe.
# Para isso, utilizamos os IDs das tabelas Heroi e Equipe.

# Homem de Ferro (id 1) -> Vingadores (id 1)
# Homem-Aranha (id 2) -> Defensores (id 2)
# Thor (id 4) -> Vingadores (id 1)
# Hulk (id 5) -> Vingadores (id 1)

cursor.executemany("""
INSERT INTO Heroi_Equipe (heroi_id, equipe_id)
VALUES (?, ?);
""", [
    (1, 1),
    (2, 2),
    (4, 1),
    (5, 1)
])

# Confirma as inserções no banco.
conexao.commit()

# Consulta as relações cadastradas.
cursor.execute("""
SELECT * FROM Heroi_Equipe;
""")

for linha in cursor.fetchall():
    print(linha)

(1, 1)
(2, 2)
(4, 1)
(5, 1)


### Consulta com `JOIN`

O `JOIN` permite combinar informações de tabelas relacionadas.

A consulta abaixo relaciona:
- `Heroi`: nome do herói;
- `Heroi_Equipe`: associação entre herói e equipe;
- `Equipe`: nome da equipe.

In [15]:
# DESAFIO 3
# Complete a consulta para mostrar o nome do herói e o nome da equipe.
# Utilize JOIN para relacionar as três tabelas.

cursor.execute("""
SELECT Heroi.nome, Equipe.nome
FROM Heroi
JOIN Heroi_Equipe ON 1
JOIN Equipe ON 1;
""")

for linha in cursor.fetchall():
    print(linha)

('Homem de Ferro', 'Vingadores')
('Homem de Ferro', 'Defensores')
('Homem de Ferro', 'Guardiões da Galáxia')
('Homem de Ferro', 'Vingadores')
('Homem de Ferro', 'Defensores')
('Homem de Ferro', 'Guardiões da Galáxia')
('Homem de Ferro', 'Vingadores')
('Homem de Ferro', 'Defensores')
('Homem de Ferro', 'Guardiões da Galáxia')
('Homem de Ferro', 'Vingadores')
('Homem de Ferro', 'Defensores')
('Homem de Ferro', 'Guardiões da Galáxia')
('Homem-Aranha', 'Vingadores')
('Homem-Aranha', 'Defensores')
('Homem-Aranha', 'Guardiões da Galáxia')
('Homem-Aranha', 'Vingadores')
('Homem-Aranha', 'Defensores')
('Homem-Aranha', 'Guardiões da Galáxia')
('Homem-Aranha', 'Vingadores')
('Homem-Aranha', 'Defensores')
('Homem-Aranha', 'Guardiões da Galáxia')
('Homem-Aranha', 'Vingadores')
('Homem-Aranha', 'Defensores')
('Homem-Aranha', 'Guardiões da Galáxia')
('Thor', 'Vingadores')
('Thor', 'Defensores')
('Thor', 'Guardiões da Galáxia')
('Thor', 'Vingadores')
('Thor', 'Defensores')
('Thor', 'Guardiões da Galá

## Etapa 11 — Salvando e baixando o banco

O arquivo `marvel.db` contém as tabelas e os registros criados durante a prática.

Para baixar uma cópia para seu computador, execute a célula abaixo. O download permite guardar o arquivo mesmo depois que o ambiente de execução do Colab for encerrado.

In [16]:
# Disponibiliza o arquivo marvel.db para download no computador.

from google.colab import files
files.download("marvel.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Encerrando a conexão

Quando terminarmos de trabalhar com o banco, podemos fechar a conexão.

In [17]:
# Fecha a conexão com o banco de dados.

conexao.close()

## Desafios extras

Crie novas consultas SQL para:

1. Exibir somente os heróis com nível de poder inferior a 90.
2. Buscar um herói pelo nome.
3. Exibir os heróis em ordem alfabética.
4. Aumentar em 5 pontos o nível de poder de um herói específico.
5. Consultar o nome de cada herói e sua equipe utilizando `JOIN`.
6. Contar quantos heróis estão cadastrados na tabela.
7. Exibir o maior e o menor nível de poder registrado.

**Não se esqueça:** comandos que alteram os dados (`INSERT`, `UPDATE` e `DELETE`) precisam ser confirmados com `conexao.commit()`.